# 04_00D Run Recommendation and Pattern Notebooks
Chay 04_12 -> 04_13.

In [ ]:
from pathlib import Path
import os
import time
import subprocess, sys
import pandas as pd
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
NOTEBOOK_DIR = PROJECT_ROOT / 'notebooks'
STOP_ON_ERROR = True
MAX_RETRY = 3
run_list = [
 '04_12_Recommendation_ALS.ipynb',
 '04_13_PatternMining_FPGrowth.ipynb',
]
VENV_PY = PROJECT_ROOT / '.venv' / 'Scripts' / 'python.exe'
RUN_PY = str(VENV_PY) if VENV_PY.exists() else sys.executable
print('Runner Python:', RUN_PY)
base_env = os.environ.copy()
base_env['PYSPARK_PYTHON'] = RUN_PY
base_env['PYSPARK_DRIVER_PYTHON'] = RUN_PY
results = []
for nb in run_list:
    src = NOTEBOOK_DIR / nb
    cmd = [RUN_PY, '-m', 'jupyter', 'nbconvert', '--to', 'notebook', '--execute', '--inplace', str(src)]
    last_msg = ''
    success = False
    for attempt in range(1, MAX_RETRY + 2):
        proc = subprocess.run(cmd, cwd=str(PROJECT_ROOT), capture_output=True, text=True, env=base_env)
        full_msg = (proc.stderr or proc.stdout or '')
        last_msg = full_msg
        if proc.returncode == 0:
            success = True
            break
        transient = ('ConnectionRefusedError' in full_msg) or ('WinError 10061' in full_msg) or ('EOFException' in full_msg) or ('ObjectInputStream' in full_msg) or ('Python worker exited unexpectedly' in full_msg)
        if transient and attempt <= MAX_RETRY:
            print(f'[RETRY {attempt}/{MAX_RETRY}]', nb)
            time.sleep(3)
            continue
        break
    if success:
        results.append((nb, 'ok', f'executed (attempt {attempt})'))
        print('[OK]', nb)
    else:
        err = (last_msg[:1200] + '\n...\n' + last_msg[-1200:]) if len(last_msg) > 2400 else last_msg
        results.append((nb, 'failed', err))
        print('[FAILED]', nb)
        if STOP_ON_ERROR:
            break
summary = pd.DataFrame(results, columns=['notebook','status','message'])
print(summary.to_string(index=False))
out = PROJECT_ROOT / 'reports' / 'run_recommendation_pattern_summary.csv'
out.parent.mkdir(parents=True, exist_ok=True)
summary.to_csv(out, index=False)
print('Saved:', out)


Runner Python: C:\Users\vanhi\Desktop\HCMUTE_TMDT\HKII_Nam_3\BigData\Nhom03_PySpark_ProjectCuoiKy\.venv\Scripts\python.exe
